# Extraction Results — Exact Match Evaluation

Micro-averaged Precision, Recall, F1 per category and overall.

**Micro-averaging** pools raw triple counts (matched, predicted, gold) before dividing — the correct approach when entries contain very few triples (1–3) and categories have different sizes.

In [1]:
import json
import os
import pandas as pd
from collections import defaultdict
import re

RUN_DIR = "results/gemini_gemini-3.1-flash-lite-preview_rdf_shacl_1"

# ── Load results from single file ────────────────────────────────────────────
results_path = os.path.join(RUN_DIR, "results.json")
with open(results_path, encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
df["category"] = df["entry_id"].str.rsplit("_", n=2).str[0]
df["n_gold"] = df["gold_triples"].apply(len)
df["n_pred"] = df["pred_triples"].apply(len)

def add_underscores(text):
    """Replace any non-alphanumeric symbols that separate words with underscores, keep numbers."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', text).strip('_')

_LITERAL_RE = re.compile(r'^[\d\.\-\+eE]+$')

def _is_literal_value(value):
    """Check if a value looks like a numeric or date literal (not an entity name)."""
    return bool(_LITERAL_RE.match(value))

def process_triple(triple):
    """Add underscores to subject and object in a triple, preserving literal values."""
    obj = triple['object']
    return {
        'subject': add_underscores(triple['subject']),
        'relation': triple['relation'],
        'object': obj if _is_literal_value(obj) else add_underscores(obj),
    }

# Apply to all gold triples
df['gold_triples'] = df['gold_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)

# Reapply to predicted triples to use updated function
df['pred_triples'] = df['pred_triples'].apply(
    lambda triples: [process_triple(t) for t in triples]
)

print(f"Loaded {len(df)} entries across {df['category'].nunique()} categories")
print(f"Gold triples: {df['n_gold'].sum()}  |  Predicted triples: {df['n_pred'].sum()}")
df[["entry_id", "category", "n_gold", "n_pred"]].head(10)

Loaded 990 entries across 57 categories
Gold triples: 1919  |  Predicted triples: 1849


,entry_id,category,n_gold,n_pred
0,1_Airport_dev_1,1_Airport,1,1
1,1_Airport_dev_2,1_Airport,1,1
2,1_Airport_dev_3,1_Airport,1,1
3,1_Airport_dev_4,1_Airport,1,1
4,1_Airport_dev_5,1_Airport,1,1
5,1_Airport_dev_6,1_Airport,1,1
6,1_Airport_dev_7,1_Airport,1,1
7,1_Airport_dev_8,1_Airport,1,1
8,1_Airport_dev_9,1_Airport,1,1
9,1_Airport_dev_10,1_Airport,1,1


In [2]:
# ── SHACL validation — rebuild data graphs from results.json ─────────────────
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, SH
from pyshacl import validate as py_validate
from collections import defaultdict
from models.data_models import EntryExtractionResult, Triple, TripleSchema
from validators.shacl_validator import OntologyIndex, build_data_graph, _label_of

RDF_ONTOLOGY_DIR = "OSKGC/ontologies/rdf"
SHACL_SHAPES_DIR = "OSKGC/ontologies/shacl_shapes"

total_violations = 0
total_conforming = 0
total_categories = 0
all_violations = []
data_graphs_by_cat = {}  # category → serialized TTL string
result_messages_by_cat = {}  # category → list of (focus, path, constraint, message)

for cat in sorted(df["category"].unique()):
    rdf_path = os.path.join(RDF_ONTOLOGY_DIR, f"{cat}.ttl")
    shacl_path = os.path.join(SHACL_SHAPES_DIR, f"{cat}_shacl.ttl")
    if not os.path.exists(rdf_path) or not os.path.exists(shacl_path):
        continue

    # Build EntryExtractionResult objects from the dataframe
    cat_df = df[df["category"] == cat]
    entries = {}
    for local_i, (_, row) in enumerate(cat_df.iterrows()):
        key = f"entry_{local_i + 1}"
        entries[key] = EntryExtractionResult(
            triples=[Triple(**t) for t in row["pred_triples"]],
            schemas=[TripleSchema(**s) for s in row["pred_schemas"]],
        )

    # Build data graph + triple map (hierarchy injected inside build_data_graph)
    with open(rdf_path, encoding="utf-8") as f:
        rdf_text = f.read()
    ont_idx = OntologyIndex(rdf_text)
    data_graph, triple_map = build_data_graph(entries, ont_idx)

    # Collect for dumping
    data_graphs_by_cat[cat] = data_graph.serialize(format="turtle")

    if len(data_graph) == 0:
        total_categories += 1
        total_conforming += 1
        continue

    shapes_graph = Graph()
    shapes_graph.parse(shacl_path, format="turtle")

    # Run pyshacl — no inference, hierarchy already materialized in data_graph
    try:
        conforms, results_graph, results_text = py_validate(
            data_graph, shacl_graph=shapes_graph,
            inference="none", abort_on_first=False,
        )
    except Exception as e:
        print(f"  ⚠ {cat}: {type(e).__name__}: {e}")
        continue

    total_categories += 1
    if conforms:
        total_conforming += 1
        continue

    result_messages_by_cat[cat] = results_text

    # Parse violations
    entry_ids = cat_df["entry_id"].tolist()
    entry_results = []
    for _, row in cat_df.iterrows():
        entry_results.append((row["pred_triples"], row.get("pred_schemas", [])))

    violations = []
    for result_node in results_graph.subjects(RDF.type, SH.ValidationResult):
        severity = results_graph.value(result_node, SH.resultSeverity)
        if severity and str(severity) != str(SH.Violation):
            continue

        focus = results_graph.value(result_node, SH.focusNode)
        path = results_graph.value(result_node, SH.resultPath)
        source = results_graph.value(result_node, SH.sourceConstraintComponent)
        messages = list(results_graph.objects(result_node, SH.resultMessage))

        focus_str = _label_of(focus) if focus else "?"
        path_str = _label_of(path) if path else None
        constraint_str = _label_of(source) if source else "unknown"
        msg = str(messages[0]) if messages else f"{constraint_str} on {path_str}"

        matches = triple_map.get((focus, path), []) if (focus and path) else []
        if matches:
            for entry_idx, t_idx in matches:
                violations.append((entry_idx, t_idx, focus_str, path_str, constraint_str, msg))
        else:
            violations.append((-1, -1, focus_str, path_str, constraint_str, msg))

    total_violations += len(violations)

    # Group by entry for display
    by_entry = defaultdict(list)
    for v in violations:
        by_entry[v[0]].append(v)

    print(f"\n{'='*80}")
    print(f"Category: {cat}  —  {len(violations)} violation(s)")
    print(f"{'='*80}")
    for entry_idx in sorted(by_entry):
        vs = by_entry[entry_idx]
        eid = entry_ids[entry_idx] if 0 <= entry_idx < len(entry_ids) else f"(unmapped, idx={entry_idx})"
        print(f"\n  Entry: {eid}")
        by_triple = defaultdict(list)
        for v in vs:
            by_triple[v[1]].append(v)
        for t_idx in sorted(by_triple):
            if 0 <= entry_idx < len(entry_results) and 0 <= t_idx < len(entry_results[entry_idx][0]):
                t = entry_results[entry_idx][0][t_idx]
                schemas = entry_results[entry_idx][1]
                s = schemas[t_idx] if t_idx < len(schemas) else {}
                print(f"    triple {t_idx+1}: ({t['subject']}, {t['relation']}, {t['object']})  "
                      f"types: ({s.get('subject','?')}, {s.get('object','?')})")
            else:
                print(f"    triple {t_idx+1}: (unmapped)")
            for v in by_triple[t_idx]:
                print(f"      violation: [{v[4]}] {v[5]}")
                all_violations.append((cat, eid, t_idx, v[4], v[3], v[5]))

print(f"\n{'─'*80}")
print(f"SHACL Summary: {total_categories} categories validated")
print(f"  Conforming : {total_conforming}")
print(f"  With violations : {total_categories - total_conforming}")
print(f"  Total violations: {total_violations}")


Category: 1_Airport  —  8 violation(s)

  Entry: 1_Airport_dev_2
    triple 1: (Belgium, leader, Philippe_of_Belgium)  types: (Country, Person)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Organisation>, <http://dbpedia.org/ontology/Person>)

  Entry: 1_Airport_dev_3
    triple 1: (Amsterdam_Airport_Schiphol, location, Haarlemmermeer)  types: (Airport, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/ArchitecturalStructure>)

  Entry: 1_Airport_dev_10
    triple 1: (Afonso_Pena_International_Airport, operatingOrganisation, Infraero)  types: (Airport, Organisation)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Organisation>, <http://dbpedia.org/ontology/Place>)

  Entry: 1_Airport_dev_12
    triple 1: (Infraero, location, Brazil)  types: (Organisation, Place)
      violatio

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x0000024D747576A0>
Traceback (most recent call last):
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\xsd_datetime.py", line 586, in parse_xsd_date
    raise ValueError("XSD Date string must contain at least two dashes")
ValueError: XSD Date string must contain at least two dashes



Category: 2_Company  —  4 violation(s)

  Entry: 2_Company_dev_3
    triple 2: (Hypermarcas, product, Drugs)  types: (Company, TopicalConcept)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Work>, <http://dbpedia.org/ontology/TopicalConcept>)

  Entry: 2_Company_dev_5
    triple 1: (Hypermarcas, foundingDate, 2001_01_01)  types: (Company, Date)
      violation: [DatatypeConstraintComponent] Value is not Literal with datatype xsd:date
    triple 2: (Hypermarcas, product, Drugs)  types: (Company, TopicalConcept)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Work>, <http://dbpedia.org/ontology/TopicalConcept>)

  Entry: 2_Company_dev_10
    triple 1: (Trane, product, HVAC)  types: (Company, TopicalConcept)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Work>, <http://dbpedia.org/ontology/TopicalConcept>)

Category: 2_F

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x0000024D747576A0>
Traceback (most recent call last):
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\xsd_datetime.py", line 586, in parse_xsd_date
    raise ValueError("XSD Date string must contain at least two dashes")
ValueError: XSD Date string must contain at least two dashes



Category: 2_MeanOfTransportation  —  15 violation(s)

  Entry: 2_MeanOfTransportation_dev_12
    triple 2: (Acura_TLX, assembly, Marysville)  types: (Automobile, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/ArchitecturalStructure>, <http://dbpedia.org/ontology/Organisation>)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/ArchitecturalStructure>, <http://dbpedia.org/ontology/Organisation>)

  Entry: 2_MeanOfTransportation_dev_13
    triple 2: (Aston_Martin_V8, assembly, United_Kingdom)  types: (Automobile, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/ArchitecturalStructure>, <http://dbpedia.org/ontology/Organisation>)
      violation: [ClassConstraintComponent] Value class is not in classes 

...


Category: 3_Building  —  20 violation(s)

  Entry: 3_Building_dev_2
    triple 2: (Cape_Town, leader, Jacob_Zuma)  types: (CapitalCity, Person)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Organisation>, <http://dbpedia.org/ontology/Person>)

  Entry: 3_Building_dev_4
    triple 1: (300_North_LaSalle, location, Chicago)  types: (Building, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/ArchitecturalStructure>, <http://dbpedia.org/ontology/Place>)

  Entry: 3_Building_dev_5
    triple 1: (Akita_Museum_of_Art, location, Akita)  types: (Museum, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/ArchitecturalStructure>, <http://dbpedia.org/ontology/Place>)

  Entry: 3_Building_dev_6
    triple 1: (Marriott_International, location, Bethesda)  types: (Company, Place)
      violation: [ClassConstraintComponent] V

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x0000024D747576A0>
Traceback (most recent call last):
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\xsd_datetime.py", line 586, in parse_xsd_date
    raise ValueError("XSD Date string must contain at least two dashes")
ValueError: XSD Date string must contain at least two dashes
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x0000024D747576A0>
Traceback (most recent call last):
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\xsd_da


Category: 3_Company  —  8 violation(s)

  Entry: 3_Company_dev_1
    triple 1: (GMA_New_Media, keyPerson, Felipe_Gozon)  types: (Company, Person)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/PersonFunction>, <http://dbpedia.org/ontology/Person>)
    triple 2: (GMA_New_Media, foundingDate, 2000_01_01)  types: (Company, Date)
      violation: [DatatypeConstraintComponent] Value is not Literal with datatype xsd:date

  Entry: 3_Company_dev_5
    triple 1: (GMA_New_Media, location, Quezon_City)  types: (Company, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/ArchitecturalStructure>, <http://dbpedia.org/ontology/Place>)

  Entry: 3_Company_dev_6
    triple 2: (AmeriGas, location, King_of_Prussia)  types: (Company, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/ArchitecturalStructure>, <http://dbpedia.org/o

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x0000024D747576A0>
Traceback (most recent call last):
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "c:\conda_envs\thesis_env\Lib\site-packages\rdflib\xsd_datetime.py", line 586, in parse_xsd_date
    raise ValueError("XSD Date string must contain at least two dashes")
ValueError: XSD Date string must contain at least two dashes



Category: 3_MeanOfTransportation  —  8 violation(s)

  Entry: 3_MeanOfTransportation_dev_4
    triple 1: (ARA_Veinticinco_de_Mayo, builder, Cammell_Laird)  types: (Ship, Organisation)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/Organisation>)

  Entry: 3_MeanOfTransportation_dev_7
    triple 1: (Pontiac_Rageous, assembly, Michigan)  types: (Automobile, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/Organisation>)

  Entry: 3_MeanOfTransportation_dev_8
    triple 1: (Alfa_Romeo_164, assembly, Italy)  types: (Automobile, Place)
      violation: [ClassConstraintComponent] Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/Organisation>)

  Entry: 3_MeanOfTransportation_dev_10
    triple 1: (A_Rosa_Luna, assembly, Rostock)  types: (Ship, Place)
    

In [5]:
print(result_messages_by_cat["2_Airport"])

Validation Report
Conforms: False
Results (12):
Constraint Violation in ClosedConstraintComponent (http://www.w3.org/ns/shacl#ClosedConstraintComponent):
	Severity: sh:Violation
	Source Shape: <http://dbpedia.org/ontology/ArchitecturalStructure>
	Focus Node: ex:entry_10__Pakistan
	Value Node: ex:entry_10__Nawaz_Sharif
	Result Path: rel:leader
	Message: Node ex:entry_10__Pakistan is closed. It cannot have value: ex:entry_10__Nawaz_Sharif
Constraint Violation in ClosedConstraintComponent (http://www.w3.org/ns/shacl#ClosedConstraintComponent):
	Severity: sh:Violation
	Source Shape: <http://dbpedia.org/ontology/ArchitecturalStructure>
	Focus Node: ex:entry_13__Madrid
	Value Node: ex:entry_13__Spain
	Result Path: rel:isPartOf
	Message: Node ex:entry_13__Madrid is closed. It cannot have value: ex:entry_13__Spain
Constraint Violation in ClosedConstraintComponent (http://www.w3.org/ns/shacl#ClosedConstraintComponent):
	Severity: sh:Violation
	Source Shape: <http://dbpedia.org/ontology/Architec

In [4]:
print(result_messages_by_cat["1_Airport"])

Validation Report
Conforms: False
Results (8):
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: <http://dbpedia.org/ontology/ArchitecturalStructure-location>
	Focus Node: ex:entry_17__Adolfo_Su_rez_Madrid_Barajas_Airport
	Value Node: ex:entry_17__Madrid
	Result Path: rel:location
	Message: Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/ArchitecturalStructure>)
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: <http://dbpedia.org/ontology/ArchitecturalStructure-location>
	Focus Node: ex:entry_30__Allama_Iqbal_International_Airport
	Value Node: ex:entry_30__Pakistan
	Result Path: rel:location
	Message: Value class is not in classes (<http://dbpedia.org/ontology/Place>, <http://dbpedia.org/ontology/ArchitecturalStructure>)
Constraint Violation in ClassCons

In [3]:
# ── Dump data graphs to TTL files ────────────────────────────────────────────
dg_dir = os.path.join(RUN_DIR, "data_graphs")
os.makedirs(dg_dir, exist_ok=True)

for cat, ttl in data_graphs_by_cat.items():
    out_path = os.path.join(dg_dir, f"{cat}.ttl")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(ttl)

print(f"Saved {len(data_graphs_by_cat)} data graphs to {dg_dir}/")

Saved 57 data graphs to results/gemini_gemini-3.1-flash-lite-preview_rdf_shacl_1\data_graphs/


In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer

FUZZY_THRESHOLD = 0.85
SYNONYM_GROUPS  = [
    {"currentTeam", "formerTeam", "club", "currentClub"},
    {"country", "isPartOf"},
]

# Load once — reused for all similarity calls
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Done.")

# ── helpers ───────────────────────────────────────────────────────────────────
def exact_match(pred: dict, gold: dict) -> bool:
    return (pred["subject"].lower()  == gold["subject"].lower()  and
            pred["relation"].lower() == gold["relation"].lower() and
            pred["object"].lower()   == gold["object"].lower())

def relations_are_synonyms(r1: str, r2: str) -> bool:
    r1, r2 = r1.lower(), r2.lower()
    return any(r1 in g and r2 in g for g in
               [{x.lower() for x in grp} for grp in SYNONYM_GROUPS])

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def fuzzy_match(pred: dict, gold: dict,
                emb_cache: dict, threshold: float = FUZZY_THRESHOLD) -> bool:
    """Relation must match exactly; subject & object matched by embedding similarity."""
    if pred["relation"].lower() != gold["relation"].lower():
        return False
    for key in ("subject", "object"):
        pv, gv = pred[key].lower(), gold[key].lower()
        if pv == gv:
            continue
        if pv not in emb_cache:
            emb_cache[pv] = embedder.encode(pv, convert_to_numpy=True)
        if gv not in emb_cache:
            emb_cache[gv] = embedder.encode(gv, convert_to_numpy=True)
        if cosine(emb_cache[pv], emb_cache[gv]) < threshold:
            return False
    return True

def synonym_match(pred: dict, gold: dict,
                  emb_cache: dict, threshold: float = FUZZY_THRESHOLD) -> bool:
    """Like fuzzy_match but also accepts synonym relations."""
    rel_ok = (pred["relation"].lower() == gold["relation"].lower() or
              relations_are_synonyms(pred["relation"], gold["relation"]))
    if not rel_ok:
        return False
    for key in ("subject", "object"):
        pv, gv = pred[key].lower(), gold[key].lower()
        if pv == gv:
            continue
        if pv not in emb_cache:
            emb_cache[pv] = embedder.encode(pv, convert_to_numpy=True)
        if gv not in emb_cache:
            emb_cache[gv] = embedder.encode(gv, convert_to_numpy=True)
        if cosine(emb_cache[pv], emb_cache[gv]) < threshold:
            return False
    return True

def count_matched_full(pred_triples: list, gold_triples: list,
                       emb_cache: dict):
    """
    Returns (exact, fuzzy_extra, synonym_extra, fuzzy_corrections, synonym_corrections).
    fuzzy_corrections  — list of (pred, gold) pairs fixed by fuzzy but not exact.
    synonym_corrections — list of (pred, gold) pairs fixed by synonym but not fuzzy.
    No double-counting: each gold triple matched at most once (best match wins).
    """
    used_exact = set(); used_fuzzy = set(); used_syn = set()
    exact_n = fuzzy_n = syn_n = 0
    fuzzy_corrections = []
    synonym_corrections = []

    # exact pass
    for p in pred_triples:
        for j, g in enumerate(gold_triples):
            if j not in used_exact and exact_match(p, g):
                exact_n += 1
                used_exact.add(j)
                break

    # fuzzy pass (only on gold not already matched exactly)
    used_fuzzy = set(used_exact)
    for p in pred_triples:
        # skip if this pred was already counted in exact
        already = any(j in used_exact and exact_match(p, gold_triples[j])
                      for j in used_exact)
        if already:
            continue
        for j, g in enumerate(gold_triples):
            if j not in used_fuzzy and fuzzy_match(p, g, emb_cache):
                fuzzy_n += 1
                used_fuzzy.add(j)
                fuzzy_corrections.append({"pred": p, "gold": g})
                break

    # synonym pass (only on gold not already matched by exact or fuzzy)
    used_syn = set(used_fuzzy)
    for p in pred_triples:
        already_exact = any(j in used_exact and exact_match(p, gold_triples[j])
                            for j in used_exact)
        already_fuzzy = any(c["pred"] == p for c in fuzzy_corrections)
        if already_exact or already_fuzzy:
            continue
        for j, g in enumerate(gold_triples):
            if j not in used_syn and synonym_match(p, g, emb_cache):
                syn_n += 1
                used_syn.add(j)
                synonym_corrections.append({"pred": p, "gold": g})
                break

    return exact_n, fuzzy_n, syn_n, fuzzy_corrections, synonym_corrections

# ── run over full dataframe ───────────────────────────────────────────────────
print("Computing matches (embedding calls may take a moment)...")
emb_cache = {}

results = df.apply(
    lambda r: count_matched_full(r["pred_triples"], r["gold_triples"], emb_cache),
    axis=1
)

df["matched_exact"]        = results.apply(lambda x: x[0])
df["matched_fuzzy_extra"]  = results.apply(lambda x: x[1])
df["matched_syn_extra"]    = results.apply(lambda x: x[2])
df["fuzzy_corrections"]    = results.apply(lambda x: x[3])
df["synonym_corrections"]  = results.apply(lambda x: x[4])

# cumulative matched columns
df["matched_fuzzy"]    = df["matched_exact"] + df["matched_fuzzy_extra"]
df["matched_synonym"]  = df["matched_fuzzy"] + df["matched_syn_extra"]

print(f"Exact matched   : {df['matched_exact'].sum()}")
print(f"+ Fuzzy extra   : {df['matched_fuzzy_extra'].sum()}  → {df['matched_fuzzy'].sum()} total")
print(f"+ Synonym extra : {df['matched_syn_extra'].sum()}  → {df['matched_synonym'].sum()} total")

Loading embedding model...
Done.
Computing matches (embedding calls may take a moment)...
Exact matched   : 1488
+ Fuzzy extra   : 41  → 1529 total
+ Synonym extra : 32  → 1561 total


In [27]:
def micro_prf(group, matched_col):
    m       = group[matched_col].sum()
    p_total = group["n_pred"].sum()
    g_total = group["n_gold"].sum()
    prec = m / p_total if p_total else 0.0
    rec  = m / g_total if g_total else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return pd.Series({"Precision": prec, "Recall": rec, "F1": f1,
                      "Matched": int(m), "Gold": int(g_total),
                      "Predicted": int(p_total), "Entries": len(group)})

cat_exact   = df.groupby("category").apply(micro_prf, matched_col="matched_exact",   include_groups=False)
cat_fuzzy   = df.groupby("category").apply(micro_prf, matched_col="matched_fuzzy",   include_groups=False)
cat_synonym = df.groupby("category").apply(micro_prf, matched_col="matched_synonym", include_groups=False)

# side-by-side F1
comparison = pd.DataFrame({
    "F1_exact":   cat_exact["F1"],
    "F1_fuzzy":   cat_fuzzy["F1"],
    "F1_synonym": cat_synonym["F1"],
    "Δ_fuzzy":    cat_fuzzy["F1"]   - cat_exact["F1"],
    "Δ_synonym":  cat_synonym["F1"] - cat_fuzzy["F1"],
    "Gold":       cat_exact["Gold"],
    "Entries":    cat_exact["Entries"],
}).sort_index()

comparison.style.format({c: "{:.3f}" for c in comparison.columns if c not in ("Gold","Entries")})

,F1_exact,F1_fuzzy,F1_synonym,Δ_fuzzy,Δ_synonym,Gold,Entries
category,,,,,,,
1_Airport,0.943,0.943,0.943,0.000,0.000,35.000000,35.000000
1_Artist,0.969,0.969,0.969,0.000,0.000,32.000000,32.000000
1_Astronaut,0.750,0.750,0.750,0.000,0.000,8.000000,8.000000
1_Athlete,0.688,0.906,0.906,0.219,0.000,32.000000,32.000000
1_Building,0.784,0.784,0.863,0.000,0.078,25.000000,25.000000
1_CelestialBody,0.947,0.947,0.947,0.000,0.000,19.000000,19.000000
1_City,0.815,0.815,0.852,0.000,0.037,27.000000,27.000000
1_ComicsCharacter,1.000,1.000,1.000,0.000,0.000,11.000000,11.000000
1_Company,0.600,0.800,0.800,0.200,0.000,10.000000,10.000000


In [28]:
def overall_prf(matched_col):
    return micro_prf(df, matched_col)

ov_exact   = overall_prf("matched_exact")
ov_fuzzy   = overall_prf("matched_fuzzy")
ov_synonym = overall_prf("matched_synonym")

for label, ov in [("EXACT", ov_exact), ("FUZZY (≥0.9 sim)", ov_fuzzy), ("SYNONYM-AWARE", ov_synonym)]:
    print(f"\n{'─'*50}")
    print(f"{label}")
    print(f"  Precision : {ov['Precision']:.4f}")
    print(f"  Recall    : {ov['Recall']:.4f}")
    print(f"  F1        : {ov['F1']:.4f}")
    print(f"  Matched   : {int(ov['Matched'])} / {int(ov['Gold'])} gold, {int(ov['Predicted'])} pred")

print(f"\n  Fuzzy  fixed {int(ov_fuzzy['Matched']  - ov_exact['Matched'])} extra triples "
      f"(F1 +{ov_fuzzy['F1']-ov_exact['F1']:.4f})")
print(f"  Synonym fixed {int(ov_synonym['Matched'] - ov_fuzzy['Matched'])} extra triples "
      f"(F1 +{ov_synonym['F1']-ov_fuzzy['F1']:.4f})")


──────────────────────────────────────────────────
EXACT
  Precision : 0.8127
  Recall    : 0.7754
  F1        : 0.7936
  Matched   : 1488 / 1919 gold, 1831 pred

──────────────────────────────────────────────────
FUZZY (≥0.9 sim)
  Precision : 0.8351
  Recall    : 0.7968
  F1        : 0.8155
  Matched   : 1529 / 1919 gold, 1831 pred

──────────────────────────────────────────────────
SYNONYM-AWARE
  Precision : 0.8525
  Recall    : 0.8134
  F1        : 0.8325
  Matched   : 1561 / 1919 gold, 1831 pred

  Fuzzy  fixed 41 extra triples (F1 +0.0219)
  Synonym fixed 32 extra triples (F1 +0.0171)


In [29]:
fuzzy_rows = df[df["matched_fuzzy_extra"] > 0][["entry_id","category","input_text","fuzzy_corrections"]]

for _, row in fuzzy_rows.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['entry_id']}  [{row['category']}]")
    print(f"Text: {row['input_text'][:120]}")
    for c in row["fuzzy_corrections"]:
        print(f"  PRED : ({c['pred']['subject']}, {c['pred']['relation']}, {c['pred']['object']})")
        print(f"  GOLD : ({c['gold']['subject']}, {c['gold']['relation']}, {c['gold']['object']})")
        print()


1_Athlete_dev_5  [1_Athlete]
Text: Alaa Abdul-Zahra played for the club Al-Zawra'a SC.
  PRED : (Alaa_Abdul_Zahra, club, Al_Zawraa_SC)
  GOLD : (Alaa_Abdul_Zahra, club, Al_Zawra_a_SC)


1_Athlete_dev_13  [1_Athlete]
Text: The Chairman of A.C. Milan is Silvio Berlusconi.
  PRED : (AC_Milan, chairman, Silvio_Berlusconi)
  GOLD : (A_C_Milan, chairman, Silvio_Berlusconi)


1_Athlete_dev_19  [1_Athlete]
Text: The United Petrotrin F.C. is playing in Palo Seco Velodrome.
  PRED : (United_Petrotrin_FC, ground, Palo_Seco_Velodrome)
  GOLD : (United_Petrotrin_F_C, ground, Palo_Seco_Velodrome)


1_Athlete_dev_20  [1_Athlete]
Text: Alessio Romagnoli plays for A.C. Milan.
  PRED : (Alessio_Romagnoli, club, AC_Milan)
  GOLD : (Alessio_Romagnoli, club, A_C_Milan)


1_Athlete_dev_21  [1_Athlete]
Text: Alan Martin played football for Accrington Stanley F.C.
  PRED : (Alan_Martin, club, Accrington_Stanley_FC)
  GOLD : (Alan_Martin, club, Accrington_Stanley_F_C)


1_Athlete_dev_23  [1_Athlete]
Text: Pal

In [30]:
syn_rows = df[df["matched_syn_extra"] > 0][["entry_id","category","input_text","synonym_corrections"]]

for _, row in syn_rows.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['entry_id']}  [{row['category']}]")
    print(f"Text: {row['input_text'][:120]}")
    for c in row["synonym_corrections"]:
        print(f"  PRED : ({c['pred']['subject']}, {c['pred']['relation']}, {c['pred']['object']})")
        print(f"  GOLD : ({c['gold']['subject']}, {c['gold']['relation']}, {c['gold']['object']})")
        print()


1_Building_dev_1  [1_Building]
Text: Ahmedabad is in India.
  PRED : (Ahmedabad, isPartOf, India)
  GOLD : (Ahmedabad, country, India)


1_Building_dev_3  [1_Building]
Text: Dublin is in the Republic of Ireland.
  PRED : (Dublin, isPartOf, Republic_of_Ireland)
  GOLD : (Dublin, country, Republic_of_Ireland)


1_City_dev_19  [1_City]
Text: Texas is in the United States.
  PRED : (Texas, isPartOf, United_States)
  GOLD : (Texas, country, United_States)


2_Airport_dev_19  [2_Airport]
Text: Al-Taqaddum Air Base serves the city of Fallujah which is in Iraq.
  PRED : (Fallujah, isPartOf, Iraq)
  GOLD : (Fallujah, country, Iraq)


2_Athlete_dev_4  [2_Athlete]
Text: Alessio Romagnoli played for the Italy national under-16 football team coached by Daniele Zoratto.
  PRED : (Alessio_Romagnoli, formerTeam, Italy_national_under_16_football_team)
  GOLD : (Alessio_Romagnoli, club, Italy_national_under_16_football_team)


2_Athlete_dev_14  [2_Athlete]
Text: Former clubs of Alessio Romagnoli includ

In [31]:
# uses matched_synonym as the final "correct" count
errors_inds = df[df["matched_synonym"] != df["n_gold"]].index.tolist()

def index_generator():
    for ind in errors_inds:
        yield ind

gen = index_generator()

In [42]:
import json as _json, os as _os, xml.etree.ElementTree as _ET

idx = next(gen)
row = df.iloc[idx]

# ── Load gold schema from ontology JSON ──────────────────────────────────────
_schema_path = _os.path.join("OSKGC", "ontologies", "json", f"{row['category']}.json")
_gold_schema = _json.load(open(_schema_path))
_gold_rel_map = {r["label"]: r for r in _gold_schema.get("relation", [])}
_used_relations = list(dict.fromkeys(t["relation"] for t in row["gold_triples"]))

# ── Load per-entry schemas from OSKGC/data/{split}/{category}.xml ────────────
# entry_id format: {category}_{split}_{n}  →  split is second-to-last segment
_parts = row["entry_id"].rsplit("_", 2)   # e.g. ["1_Airport", "dev", "1"]
_split = _parts[1] if len(_parts) == 3 else "dev"
_xml_path = _os.path.join("OSKGC", "data", _split, f"{row['category']}.xml")
_entry_schemas = []
if _os.path.exists(_xml_path):
    _tree = _ET.parse(_xml_path)
    for _entry in _tree.findall(".//entry"):
        if _entry.get("id") == row["entry_id"]:
            for _s in _entry.findall(".//schema"):
                _entry_schemas.append({
                    "subject": _s.findtext("sub", ""),
                    "relation": _s.findtext("rel", ""),
                    "object":  _s.findtext("obj", ""),
                })
            break

print("=" * 80)
print(f"Index: {idx} | Entry ID: {row['entry_id']}")
print(f"Category: {row['category']} | "
      f"Exact: {row['matched_exact']}/{row['n_gold']} | "
      f"Fuzzy: {row['matched_fuzzy']}/{row['n_gold']} | "
      f"Synonym: {row['matched_synonym']}/{row['n_gold']}")
print("-" * 80)
print(f"Input: {row['input_text']}")
print("-" * 80)
print("GOLD TRIPLES:")
for i, t in enumerate(row["gold_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("-" * 80)
print("PRED TRIPLES:")
for i, t in enumerate(row["pred_triples"], 1):
    print(f"  {i}. ({t['subject']}, {t['relation']}, {t['object']})")
print("-" * 80)
print("PRED SCHEMA (predicted subject/object types):")
for i, s in enumerate(row["pred_schemas"], 1):
    print(f"  {i}. subject={s['subject']}  object={s['object']}")
print("-" * 80)
print("GOLD SCHEMA — per-entry (from data XML):")
if _entry_schemas:
    for i, s in enumerate(_entry_schemas, 1):
        print(f"  {i}. [{s['relation']}]  subject={s['subject']}  object={s['object']}")
else:
    print("  (not found)")
print("-" * 80)
print("GOLD SCHEMA — ontology (domain/range for gold relations):")
for i, rel in enumerate(_used_relations, 1):
    r = _gold_rel_map.get(rel)
    if r:
        print(f"  {i}. [{rel}]  domain={r['domain']}  range={r['range']}")
    else:
        print(f"  {i}. [{rel}]  (not found in ontology)")
print("=" * 80)


Index: 117 | Entry ID: 1_Building_dev_11
Category: 1_Building | Exact: 0/1 | Fuzzy: 0/1 | Synonym: 0/1
--------------------------------------------------------------------------------
Input: London is led via the European Parliament.
--------------------------------------------------------------------------------
GOLD TRIPLES:
  1. (London, leaderTitle, European_Parliament)
--------------------------------------------------------------------------------
PRED TRIPLES:
  1. (London, governingBody, European_Parliament)
--------------------------------------------------------------------------------
PRED SCHEMA (predicted subject/object types):
  1. subject=City  object=Legislature
--------------------------------------------------------------------------------
GOLD SCHEMA — per-entry (from data XML):
  1. [leaderTitle]  subject=CapitalCity  object=Legislature
--------------------------------------------------------------------------------
GOLD SCHEMA — ontology (domain/range for gold rela

In [40]:
def calculate_embedding_similarity(text1: str, text2: str) -> float:
    """Calculate cosine similarity between embeddings of two texts."""
    emb1 = embedder.encode(text1, convert_to_numpy=True)
    emb2 = embedder.encode(text2, convert_to_numpy=True)
    return cosine(emb1, emb2)

similarity = calculate_embedding_similarity("Salvadoran_Primera_Divisi_n", "Salvadoran_Primera_Division")
print(f"Similarity: {similarity:.4f}")

Similarity: 0.8512
